# LUAD Step 2: Mutation Features and Position Masks

This notebook inspects the LUAD mutation-adjusted windows and derives the two mask types used by the downstream sequence workflow:

- a per-position validity mask that marks real DNA bases versus `N` padding
- a centered mutation-site mask that marks the edited base location in each 101-mer window

It is intentionally read-only and does not overwrite any processed files.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List

import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "luad_processed").exists():
    PROJECT_DIR = Path(r"C:/Users/LAPTOP/Downloads/luad_reserch_code")

PROCESSED_DIR = PROJECT_DIR / "luad_processed"
PRIMARY_INPUT = PROCESSED_DIR / "luad_mutation_adjusted_windows_v3.csv"
FALLBACK_INPUT = PROCESSED_DIR / "luad_mutation_adjusted_windows.csv"
INPUT_PATH = PRIMARY_INPUT if PRIMARY_INPUT.is_file() else FALLBACK_INPUT

WINDOW_LENGTH = 101
CENTER_INDEX = WINDOW_LENGTH // 2

print(f"Project directory: {PROJECT_DIR}")
print(f"Input file: {INPUT_PATH}")
print(f"Window length: {WINDOW_LENGTH}")
print(f"Mutation center index: {CENTER_INDEX}")

In [ ]:
df = pd.read_csv(INPUT_PATH, dtype=str, low_memory=False)

required_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "GENE_SYMBOL",
    "MUTATED_DNA_WINDOW",
    "WINDOW_LENGTH",
    "MUTATION_INDEX_IN_WINDOW",
    "MUTATED_WINDOW_VALID",
    "SEQUENCE_METHOD",
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Loaded rows:", len(df))
print("Columns present:", len(df.columns))
print("Unique cohorts:", df["COHORT"].nunique())
print("Valid mutated windows:", int(df["MUTATED_WINDOW_VALID"].fillna("0").astype(str).isin({"1", "true", "True", "yes", "YES"}).sum()))

In [ ]:
def valid_base_mask(window: str) -> List[int]:
    text = str(window).strip().upper()
    return [1 if base in {"A", "C", "G", "T"} else 0 for base in text]


def mutation_site_mask(index: int, length: int = WINDOW_LENGTH) -> List[int]:
    mask = [0] * length
    if 0 <= index < length:
        mask[index] = 1
    return mask


feature_df = df[[
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "GENE_SYMBOL",
    "TRANSCRIPT_ID_MATCHED",
    "HGVSc_CLEAN",
    "SEQUENCE_METHOD",
    "MUTATED_DNA_WINDOW",
    "WINDOW_LENGTH",
    "MUTATION_INDEX_IN_WINDOW",
    "MUTATED_WINDOW_VALID",
]].copy()

feature_df["window_length_int"] = pd.to_numeric(feature_df["WINDOW_LENGTH"], errors="coerce")
feature_df["mutation_index"] = pd.to_numeric(feature_df["MUTATION_INDEX_IN_WINDOW"], errors="coerce")
feature_df["sequence_length"] = feature_df["MUTATED_DNA_WINDOW"].fillna("").map(len)
feature_df["position_mask"] = feature_df["MUTATED_DNA_WINDOW"].map(valid_base_mask)
feature_df["mutation_site_mask"] = feature_df["mutation_index"].fillna(CENTER_INDEX).astype(int).map(mutation_site_mask)
feature_df["position_mask_sum"] = feature_df["position_mask"].map(sum)
feature_df["padding_bases"] = feature_df["position_mask"].map(lambda mask: len(mask) - sum(mask))
feature_df["mutation_site_is_center"] = feature_df["mutation_index"].fillna(CENTER_INDEX).astype(int).eq(CENTER_INDEX)

feature_df["position_mask_json"] = feature_df["position_mask"].map(json.dumps)
feature_df["mutation_site_mask_json"] = feature_df["mutation_site_mask"].map(json.dumps)

feature_df.head(3)

In [ ]:
valid_window_mask = feature_df["sequence_length"].eq(WINDOW_LENGTH) & feature_df["position_mask_sum"].eq(WINDOW_LENGTH)
center_mask = feature_df["mutation_site_is_center"]

print("Position-mask validation:", "PASS" if valid_window_mask.all() else "FAIL")
print("Centered mutation-site validation:", "PASS" if center_mask.all() else "FAIL")
print("Rows with any padding:", int((feature_df["padding_bases"] > 0).sum()))
print("Average padding bases per window:", round(feature_df["padding_bases"].mean(), 2))
print("Maximum padding bases in a window:", int(feature_df["padding_bases"].max()))
print("Unique sequence methods:", sorted(feature_df["SEQUENCE_METHOD"].fillna("").unique().tolist()))

In [ ]:
preview_columns = [
    "COHORT",
    "SAMPLE_ID",
    "GENE_SYMBOL",
    "SEQUENCE_METHOD",
    "position_mask_sum",
    "padding_bases",
    "mutation_site_is_center",
    "MUTATED_DNA_WINDOW",
]

preview = feature_df[preview_columns].copy()
preview["MUTATED_DNA_WINDOW"] = preview["MUTATED_DNA_WINDOW"].str.slice(0, 80) + "..."
preview.head(10)

## Notes

The two derived masks are complementary:

- `position_mask` keeps track of the usable DNA positions in each window.
- `mutation_site_mask` keeps the focal mutation location explicit for downstream models.

The notebook stops here so the feature construction remains transparent and easy to inspect before any model-specific preprocessing.